# RE_402 · Quants estils hi ha en un número? · resolt

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes a l'equip de recomanacions d'una plataforma de música. Spotify dona 12 números per a cada
cançó (ballable, energia, acústica, tempo...). El cap de producte vol fer «estils» automàtics a
partir d'aquests números, sense fer servir el gènere que diu la playlist.

**La pregunta:** quanta informació diferent hi ha de debò en aquestes 12 columnes? I els grups que surten
d'agrupar-les, s'assemblen als gèneres musicals?

L'enunciat complet, amb el diccionari de columnes, és a [RE_402](../RE_402_spotify_estils.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
df = pd.read_csv(URL)
print(df.shape)
df.head()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

AUDIO = ["danceability", "energy", "key", "loudness", "mode", "speechiness", "acousticness",
         "instrumentalness", "liveness", "valence", "tempo", "duration_ms"]

## 1.1 · Les 12 columnes

Mira el rang (mínim, màxim, desviació) de les 12 columnes d'àudio. Quines estan en una escala
molt diferent de les altres? Quines són, en realitat, categories escrites amb números?

In [ ]:
X = df[AUDIO]
assert X.shape[1] == 12 and X.isna().sum().sum() == 0
pd.DataFrame({"min": X.min(), "max": X.max(), "desv": X.std(), "diferents": X.nunique()}).round(2)

Vuit columnes van de 0 a 1. `loudness` (dB, negatiu), `tempo` (bpm) i sobretot `duration_ms`
(desviació de 60.000) són en escales molt diferents. `key` (0-11) i `mode` (0/1) són **categories
escrites amb números**: la tonalitat 11 no és «més» que la 1.

## 1.2 · Una cançó, una fila?

La mateixa cançó (`track_id`) pot sortir més d'una vegada? Per què passaria en aquest dataset?
Decideix com et quedes amb una fila per cançó, i explica què passa amb el gènere de les que surten
en playlists de gèneres diferents.

In [ ]:
print("track_id repetits:", df["track_id"].duplicated().sum())
gen = df.groupby("track_id")["playlist_genre"].nunique()
print("cançons en playlists de més d'un gènere:", (gen > 1).sum())
df = df.drop_duplicates(subset="track_id").reset_index(drop=True)
assert df["track_id"].is_unique
df.shape

La mateixa cançó surt a diverses playlists (4.477 files repetides). 1.687 cançons surten en playlists de
gèneres diferents: el gènere és de la **playlist**, no de la cançó. Ens quedem amb la primera
aparició (28.356 cançons) i som conscients que el gènere de les cançons compartides és arbitrari.

## 2.1 · PCA sense escalar

Fes un PCA de les 12 columnes **tal com venen**. Quina part de la variància explica la primera
component? Mira'n els pesos (`components_[0]`): quina columna hi mana? Què vol dir això?

In [ ]:
X = df[AUDIO]
pca_cru = PCA().fit(X)
assert np.isclose(pca_cru.explained_variance_ratio_.sum(), 1)
print("PC1 explica:", pca_cru.explained_variance_ratio_[0].round(6))
pd.Series(pca_cru.components_[0], index=AUDIO).round(4)

La primera component explica pràcticament el **100 %** de la variància i és pràcticament només
`duration_ms`. No és que la durada sigui la característica més important de la música: és que es
mesura en **mil·lisegons** i la seva variància és milions de vegades la de les altres. Un PCA
sense escalar respon a les unitats.

## 2.2 · PCA a mà

Estandarditza les dades i fes el PCA **a mà**: la matriu de covariància, els valors i vectors
propis, ordenats de més gran a més petit. Compara la variància explicada amb la de `PCA` de
scikit-learn sobre les mateixes dades.

In [ ]:
scaler = StandardScaler().fit(X)
Z = scaler.transform(X)
C = np.cov(Z, rowvar=False)
valors, vectors = np.linalg.eigh(C)
ordre = np.argsort(valors)[::-1]
valors, vectors = valors[ordre], vectors[:, ordre]
ratio = valors / valors.sum()

pca = PCA().fit(Z)
assert np.allclose(ratio, pca.explained_variance_ratio_)
assert np.allclose(np.abs(vectors.T), np.abs(pca.components_), atol=1e-6)
print("signes girats:", int((np.sign(vectors.T[:, 0]) != np.sign(pca.components_[:, 0])).sum()), "de 12")
ratio.round(3)

## 2.3 · Quantes dimensions de debò

Amb les dades estandarditzades: quantes components calen per explicar el 80 % de la variància?
Dibuixa la variància explicada acumulada. Què respondries al cap de producte sobre «quanta
informació diferent» hi ha?

In [ ]:
acum = np.cumsum(pca.explained_variance_ratio_)
assert (np.diff(acum) > 0).all() and np.isclose(acum[-1], 1)
n80 = np.argmax(acum >= 0.8) + 1
assert n80 == 8
plt.plot(range(1, 13), acum, marker="o"); plt.axhline(0.8, ls="--", c="grey")
plt.xlabel("components"); plt.ylabel("variància acumulada")
print("components per al 80 %:", n80, "| PC1:", acum[0].round(3))

Calen **8 de 12** components per arribar al 80 %, i la primera només n'explica un 18 %. Les
columnes d'àudio estan **poc correlacionades**: cadascuna diu una cosa bastant diferent. Hi ha
molta informació «diferent», però no s'organitza en pocs eixos.

## 2.4 · Grups contra gèneres

Agafa una mostra de 5.000 cançons (amb `random_state`), estandarditza i fes k-means per a
diversos valors de k. Tria'n un amb el colze i la silueta. Després, només després, creua els grups
amb `playlist_genre`. S'hi assemblen?

In [ ]:
mostra = df.sample(5000, random_state=0)
sc = StandardScaler().fit(mostra[AUDIO]); Zm = sc.transform(mostra[AUDIO])
files = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Zm)
    files.append({"k": k, "inercia": km.inertia_, "silueta": silhouette_score(Zm, km.labels_),
                  "ARI_genere": adjusted_rand_score(mostra["playlist_genre"], km.labels_)})
r = pd.DataFrame(files).set_index("k")
assert r["silueta"].max() < 0.2
r.round(3)

In [ ]:
km = KMeans(n_clusters=6, n_init=10, random_state=0).fit(Zm)
assert np.bincount(km.labels_).sum() == 5000
tc = pd.crosstab(km.labels_, mostra["playlist_genre"], normalize="index").round(2)
tc

La silueta no passa de 0,15 per a cap k: **no hi ha grups ben separats**, és un núvol continu. Amb
k = 6 (tants grups com gèneres) cap grup és un gènere: la proporció màxima d'un gènere en un grup
és lluny d'1, i l'ARI és de l'ordre de 0,06 (0 = atzar). El més semblant a un gènere és un grup
amb un 58 % de rap (molta `speechiness`) i un altre amb un 57 % d'edm (molt instrumental); la
resta barregen tots els gèneres. Els gèneres de playlist són
etiquetes culturals que el so només explica en part.

## 2.5 · Què és cada grup

Descriu dos o tres dels teus grups amb paraules (no amb noms de gènere): com sonen? Fes-ho a
partir de la mitjana de les columnes d'àudio en unitats originals.

In [ ]:
perfil = pd.DataFrame(sc.inverse_transform(km.cluster_centers_), columns=AUDIO)
perfil["cançons"] = np.bincount(km.labels_)
perfil[["danceability", "energy", "acousticness", "speechiness", "valence", "instrumentalness", "tempo", "cançons"]].round(2)

Lectura (pot canviar amb la llavor): un grup **acústic i tranquil** (acousticness alta, energia
baixa), un de **parlat** (speechiness alta, el rap), un d'**instrumental** (instrumentalness
alta: electrònica sense veu), i grups de pop «alegre i ballable» contra «enèrgic i fosc» que es
distingeixen sobretot per `valence`.

## Conclusió

Hi ha molta informació diferent, però no s'organitza en pocs eixos: amb les dades
estandarditzades la primera component explica només un 18 % i calen 8 de 12 components per arribar
al 80 %. Sense escalar, el PCA diu el contrari (una component amb gairebé el 100 %), però només
perquè `duration_ms` va en mil·lisegons. Els grups de k-means no són estils clars: la silueta no
passa de 0,15 per a cap k, i amb 6 grups cap no és un gènere (ARI ≈ 0,06). El gènere és de la
playlist, una etiqueta cultural que el so només explica en part; el més semblant a un gènere són un
grup de veu parlada (rap) i un d'instrumental (edm).